# 01 — Prepare Data
Subset the locally downloaded FLIR v27 dataset and organize it for YOLOv8 training.

In [ ]:
# Cell 1 — Config: EDIT THESE PATHS BEFORE RUNNING

RAW_DATASET_PATH = "C:/path/to/your/flir-data-set-27"  # <-- change this to your folder
OUTPUT_PATH      = "../data/processed"

TRAIN_SIZE = 2400
VAL_SIZE   = 400
TEST_SIZE  = 200

In [ ]:
# Cell 2 — Install dependencies
%pip install ultralytics opencv-python pillow matplotlib -q

In [ ]:
# Cell 3 — Explore raw dataset: count images in each split
import os

img_exts = (".jpg", ".jpeg", ".png", ".bmp")

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(RAW_DATASET_PATH, split, "images")
    lbl_dir = os.path.join(RAW_DATASET_PATH, split, "labels")

    if not os.path.exists(img_dir):
        print(f"[{split}] ❌ Not found: {img_dir}")
        continue

    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(img_exts)]
    lbls = [f for f in os.listdir(lbl_dir) if f.endswith(".txt")] if os.path.exists(lbl_dir) else []

    print(f"[{split}]  images: {len(imgs):>6,}   labels: {len(lbls):>6,}")

In [ ]:
# Cell 4 — Create subset: sample images that have matching labels, copy to OUTPUT_PATH
import random
import shutil

random.seed(42)

splits = {
    "train": TRAIN_SIZE,
    "valid": VAL_SIZE,
    "test" : TEST_SIZE,
}

for split, size in splits.items():
    src_img = os.path.join(RAW_DATASET_PATH, split, "images")
    src_lbl = os.path.join(RAW_DATASET_PATH, split, "labels")
    dst_img = os.path.join(OUTPUT_PATH, split, "images")
    dst_lbl = os.path.join(OUTPUT_PATH, split, "labels")

    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    # Only keep images that have a matching .txt label file
    all_imgs = [f for f in os.listdir(src_img) if f.lower().endswith(img_exts)]
    paired   = [
        f for f in all_imgs
        if os.path.exists(os.path.join(src_lbl, os.path.splitext(f)[0] + ".txt"))
    ]

    selected = random.sample(paired, min(size, len(paired)))

    for fname in selected:
        stem = os.path.splitext(fname)[0]
        shutil.copy2(os.path.join(src_img, fname),           os.path.join(dst_img, fname))
        shutil.copy2(os.path.join(src_lbl, stem + ".txt"),   os.path.join(dst_lbl, stem + ".txt"))

    print(f"[{split}]  copied {len(selected)} image+label pairs")

In [ ]:
# Cell 5 — Write dataset.yaml with absolute paths
import yaml

abs_output = os.path.abspath(OUTPUT_PATH)

config = {
    "path" : abs_output,
    "train": "train/images",
    "val"  : "valid/images",
    "test" : "test/images",
    "nc"   : 4,
    "names": ["person", "car", "bicycle", "dog"],
}

yaml_path = os.path.join(OUTPUT_PATH, "dataset.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Written: {os.path.abspath(yaml_path)}")

In [ ]:
# Cell 6 — Verify: print final counts and yaml contents
print("Final counts in processed dataset:")
print("-" * 35)

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(OUTPUT_PATH, split, "images")
    lbl_dir = os.path.join(OUTPUT_PATH, split, "labels")

    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(img_exts)]
    lbls = [f for f in os.listdir(lbl_dir) if f.endswith(".txt")]

    # Warn if any mismatch
    img_stems = {os.path.splitext(f)[0] for f in imgs}
    lbl_stems = {os.path.splitext(f)[0] for f in lbls}
    missing   = img_stems - lbl_stems

    status = "✅" if not missing else f"⚠️  ({len(missing)} missing labels)"
    print(f"[{split}]  images={len(imgs)}  labels={len(lbls)}  {status}")

print()
print("dataset.yaml contents:")
print("-" * 35)
print(open(yaml_path).read())

In [ ]:
# Cell 7 — Visualize: 6 random train images with GT bounding boxes in a 2x3 grid
import cv2
import numpy as np
import matplotlib.pyplot as plt

CLASS_NAMES = ["person", "car", "bicycle", "dog"]
COLORS      = [(255, 80, 80), (80, 150, 255), (80, 220, 100), (255, 200, 50)]

train_img_dir = os.path.join(OUTPUT_PATH, "train", "images")
train_lbl_dir = os.path.join(OUTPUT_PATH, "train", "labels")

all_imgs = [f for f in os.listdir(train_img_dir) if f.lower().endswith(img_exts)]
samples  = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("Sample Processed Training Images — Ground Truth Boxes", fontsize=12)

for ax, fname in zip(axes.flat, samples):
    img_path = os.path.join(train_img_dir, fname)
    lbl_path = os.path.join(train_lbl_dir, os.path.splitext(fname)[0] + ".txt")

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # Draw each box from YOLO label file
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                color = COLORS[cls_id] if cls_id < 4 else (200, 200, 200)
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = CLASS_NAMES[cls_id] if cls_id < 4 else str(cls_id)
                cv2.putText(img, label, (x1, max(y1 - 4, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    ax.imshow(img)
    ax.set_title(fname[:28], fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.savefig("../data/sample_images.png", dpi=150)
plt.show()
print("Saved: data/sample_images.png")